# 01 Generate DIM Data
This notebook generates synthetic data for Products, Customers, and Employees compatible with KiotViet.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer
from faker import Faker

fake = Faker('vi_VN')

# Setup paths
DIRS = ['../data/out/excel', '../data/out/csv', '../data/warehouse']
for d in DIRS: os.makedirs(d, exist_ok=True)

# Load localized config
with open('../data/warehouse/dim_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

print("Setup and Config loaded.")

Setup and Config loaded.


## 1. Load Seed Data & Define Single-Table Metadata

In [2]:
df_prod_seed = pd.read_excel('../templates/MauFileSanPham.xlsx', header=0)
df_cust_seed = pd.read_excel('../templates/MauFileKhachHang.xlsx', sheet_name='CustomerTemplate', header=0)
df_empl_seed = pd.read_excel('../templates/MauFileImportNhanVien.xlsx', header=0)

pm = SingleTableMetadata()
pm.detect_from_dataframe(df_prod_seed)
pm.update_column(column_name='Mã hàng', sdtype='id')
pm.update_column(column_name='Tên hàng', sdtype='categorical')
pm.update_column(column_name='Thương hiệu', sdtype='categorical')
pm.set_primary_key(column_name='Mã hàng')

cm = SingleTableMetadata()
cm.detect_from_dataframe(df_cust_seed)
cm.update_column(column_name='Mã khách hàng', sdtype='id')
cm.update_column(column_name='Tên khách hàng', sdtype='name')
cm.update_column(column_name='Điện thoại', sdtype='phone_number')
cm.update_column(column_name='Địa chỉ', sdtype='address')
cm.update_column(column_name='Loại khách', sdtype='categorical')
cm.update_column(column_name='Khu vực giao hàng', sdtype='categorical')
cm.update_column(column_name='Phường/Xã', sdtype='categorical')
cm.set_primary_key(column_name='Mã khách hàng')

em = SingleTableMetadata()
em.detect_from_dataframe(df_empl_seed)
em.update_column(column_name='Mã nhân viên', sdtype='id')
em.update_column(column_name='Tên nhân viên (*)', sdtype='name')
em.update_column(column_name='Số điện thoại (*)', sdtype='phone_number')
em.update_column(column_name='Loại lương', sdtype='categorical')
em.update_column(column_name='Chức danh', sdtype='categorical')
em.set_primary_key(column_name='Mã nhân viên')

print("Single-table Metadata initialized for each DIM.")

Single-table Metadata initialized for each DIM.


/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/metadata/single_table.py:835: UserWarning: There is an existing primary key 'Mã hàng'. This key will be removed.
  warnings.warn(
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/metadata/single_table.py:835: UserWarning: There is an existing primary key 'Email'. This key will be removed.
  warnings.warn(
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/metadata/single_table.py:835: UserWarning: There is an existing primary key 'Email'. This key will be removed.
  warnings.warn(


## 2. Independent Model Generation

In [3]:
prod_synthesizer = GaussianCopulaSynthesizer(pm)
prod_synthesizer.fit(df_prod_seed)

cust_synthesizer = GaussianCopulaSynthesizer(cm)
cust_synthesizer.fit(df_cust_seed)

empl_synthesizer = GaussianCopulaSynthesizer(em)
empl_synthesizer.fit(df_empl_seed)

def generate_dim_data(counts={'products': 100, 'customers': 50, 'employees': 10}):
    df_p = prod_synthesizer.sample(num_rows=counts['products'])
    for i, _ in df_p.iterrows():
        cat = np.random.choice(config['grocery_categories'])
        sub_cat = np.random.choice(cat['sub_categories'])
        brand = np.random.choice(config['brands'])
        df_p.at[i, 'Nhóm hàng(3 Cấp)'] = f"{cat['name']} >> {sub_cat}"
        df_p.at[i, 'Thương hiệu'] = brand
        df_p.at[i, 'Tên hàng'] = f"{sub_cat} {brand} {fake.word()}"
        if df_p.at[i, 'Giá vốn'] > df_p.at[i, 'Giá bán']:
            df_p.at[i, 'Giá bán'] = df_p.at[i, 'Giá vốn'] * 1.2
    
    df_c = cust_synthesizer.sample(num_rows=counts['customers'])
    for i, _ in df_c.iterrows():
        dist = np.random.choice(config['locations']['districts'])
        df_c.at[i, 'Khu vực giao hàng'] = f"{config['locations']['city']} - Quận {dist}"
        df_c.at[i, 'Địa chỉ'] = fake.address()
        
    df_e = empl_synthesizer.sample(num_rows=counts['employees'])
    
    return {'products': df_p, 'customers': df_c, 'employees': df_e}

gen_data = generate_dim_data()
print("Generation complete.")

/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/cuongpt/Workspaces/gjs-smart-retail-code/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in futu

Generation complete.


## 3. Export to Snapshots and Templates

In [4]:
for name, df in gen_data.items():
    df.to_csv(f'../data/out/csv/dim_{name}.csv', index=False)
    df.to_excel(f'../data/out/excel/DIM_{name.upper()}_Import.xlsx', index=False)

print("All DIM snapshots exported to data/out/")

All DIM snapshots exported to data/out/
